## Harmony Py Library
### Batch Processing Example

This notebook demonstrates submitting many Harmony requests at once and processing them
as a batch, without having to track each job's completion individually. This is useful
when you have a large number of independent requests (e.g. one per granule, date range, or
region of interest) that you want to send quickly and collect results from once they're all
done.

Three methods make this possible:

* `submit_batch()` - submits a list of `Request` objects concurrently and returns a list of
  job IDs, in the same order as the requests.
* `wait_for_batch()` - waits for a list of job IDs to each reach a terminal state, polling
  them concurrently, and reports back which jobs succeeded and which failed.
* `download_batch()` - downloads the output files for a list of job IDs concurrently.

In [ ]:
import sys
import helper
helper.install_project_and_dependencies('..')

import datetime as dt
from harmony import BBox, Client, Collection, Request, Environment

#### Build a batch of requests

Here we build several independent subset requests against the same collection, each for a
different time range. In practice these might instead vary by spatial area, granule, or any
other criteria.

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage

collection = Collection(id='C1234088182-EEDTEST')

time_ranges = [
    {
        'start': dt.datetime(2020, 1, 1),
        'stop': dt.datetime(2020, 1, 5),
    },
    {
        'start': dt.datetime(2020, 1, 1),
        'stop': dt.datetime(2020, 1, 5),
    },
    {
        'start': dt.datetime(2020, 2, 1),
        'stop': dt.datetime(2020, 2, 5),
    },
]

requests = [Request(collection=collection, temporal=time_range, labels=['batch_submit']) for time_range in time_ranges]

##### Add a request that we know will fail due to an access issue downloading the file.

In [ ]:
failing_request = Request(
    collection=Collection(id='C1234724470-POCLOUD'),
    spatial=BBox(-45,0,45,90),
    granule_id='G1236546414-POCLOUD'
)
requests.append(failing_request)

#### Submit the batch

`submit_batch()` submits every request concurrently rather than waiting for each one to be
accepted before submitting the next, and returns the resulting job IDs in the same order as
`requests`.

In [ ]:
job_ids = harmony_client.submit_batch(requests)
job_ids

#### Wait for the batch to finish

`wait_for_batch()` polls every job in the batch concurrently until each one reaches a
terminal state (`successful`, `failed`, `canceled`, `complete_with_errors`, or `paused`), and
returns a `BatchStatus` with a count and a list of job IDs for each status.

In [ ]:
batch_status = harmony_client.wait_for_batch(job_ids, show_progress=True)
batch_status.counts

##### Get the list of successful job ids.

In [ ]:
batch_status.job_ids["successful"]

#### Inspect any failures

For jobs that didn't succeed, `status()` can be used to see why.

In [ ]:
for job_id in batch_status.job_ids['failed']:
    status = harmony_client.status(job_id)
    print(job_id, status['status'], status['message'])

#### Download the outputs

`download_batch()` downloads the output files for every job in the batch concurrently and
returns a dict mapping each job ID to a list of `Future`s, one per output file. Calling
`.result()` on a `Future` blocks until that particular file finishes downloading and returns
its local filename.

It's safe to pass every job ID from the batch, regardless of status — a job with no output
files (e.g. one that failed or has no results) just maps to an empty list.

In [ ]:
downloads = harmony_client.download_batch(job_ids, directory='/tmp', overwrite=True)

for job_id, futures in downloads.items():
    filenames = [f.result() for f in futures]
    print(f'{job_id}: {filenames}')